# Penjelasan Teknis: Semi-Supervised Learning dengan Pseudo-Labeling

Berdasarkan metodologi yang dipopulerkan oleh Lee (2013) dalam artikel *"Pseudo-Label: The Simple and Efficient Semi-Supervised Learning Method for Deep Neural Networks"*, pendekatan ini bertujuan untuk memanfaatkan data yang tidak berlabel (*unlabeled data*) guna meningkatkan performa model ketika data berlabel (*labeled data*) jumlahnya terbatas.

### Alur Kerja Utama:
1.  **Pelatihan Teacher Model:** Model dilatih menggunakan dataset yang telah dilabeli secara manual (Supervised Learning) via Label-Studio.
2.  **Prediksi Data Mentah:** Model yang telah dilatih kemudian digunakan untuk memprediksi label dari dataset yang tidak memiliki label.
3.  **Seleksi Threshold (Confidence Filtering):** Tidak semua hasil prediksi otomatis diambil. Hanya data dengan probabilitas prediksi (confidence score) tertinggi (misalnya > 90%) yang diubah menjadi 'Pseudo-Labels'.
4.  **Penggabungan Dataset:** Pseudo-labels tersebut digabungkan dengan label manual asli untuk memperbesar ukuran dataset.
5.  **Pelatihan Ulang:** Dataset gabungan ini digunakan untuk melatih model akhir (Student Model) atau memperbaiki model yang ada agar memiliki generalisasi yang lebih baik.

### References:
*   **Artikel Pseudo-Labeling:** [Review: Pseudo-Label — The Simple and Efficient Semi-Supervised Learning Method for Deep Neural Networks](https://sh-tsang.medium.com/review-pseudo-label-the-simple-and-efficient-semi-supervised-learning-method-for-deep-neural-aa11b424ac29)
*   **Label Studio GitHub:** [Human-in-the-loop data labeling tool](https://github.com/HumanSignal/label-studio)

### 1. Library Imports

In [20]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
from scipy.special import softmax

### 2. Data Preparation & Cleaning

In [22]:
df_manual = pd.read_csv('/content/manual_labelled_data.csv')
df_unlabeled = pd.read_csv('/content/sisa_data_unlabeled_9000.csv')

df_manual = df_manual.rename(columns={'text': 'cleaned_content', 'sentiment': 'label'})

def clean_df_text(df):
    df = df.dropna(subset=['cleaned_content']).copy()
    df['cleaned_content'] = df['cleaned_content'].astype(str)
    return df

df_manual = clean_df_text(df_manual)
df_unlabeled = clean_df_text(df_unlabeled)

label_map = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
df_manual['label_idx'] = df_manual['label'].map(label_map)

df_valid = df_manual.dropna(subset=['label_idx']).copy()
df_valid['label'] = df_valid['label_idx'].astype(int)
df_manual = df_valid[['cleaned_content', 'label']]

display(df_manual.head())

,cleaned_content,label
0,bintang dulu karena mau kirim riwayat transaks...,0
1,bagus dan sangat membantu,2
2,luar biasa,2
3,tambah tidak jelas ini aplikasi nya kok di sur...,0
4,okeh,2


(976, 2)

### 3. Teacher Model Training (IndoBERT)

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3).to(device)

def tokenize_fn(batch):
    return tokenizer(batch["cleaned_content"], padding="max_length", truncation=True, max_length=128)

train_df, val_df = train_test_split(df_manual, test_size=0.2, random_state=42, stratify=df_manual['label'])

train_dataset = Dataset.from_pandas(train_df).map(tokenize_fn, batched=True)
val_dataset = Dataset.from_pandas(val_df).map(tokenize_fn, batched=True)

training_args = TrainingArguments(
    output_dir="./teacher_checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=10
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()
trainer.save_model("./saved_teacher_model")
tokenizer.save_pretrained("./saved_teacher_model")

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/780 [00:00<?, ? examples/s]

Map:   0%|          | 0/196 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_teacher_model/tokenizer_config.json',
 './saved_teacher_model/tokenizer.json')

### 4. Pseudo-Labeling & Dataset Merging

In [24]:
unlabeled_dataset = Dataset.from_pandas(df_unlabeled).map(tokenize_fn, batched=True)

teacher_pred_trainer = Trainer(model=AutoModelForSequenceClassification.from_pretrained("./saved_teacher_model").to(device))
raw_predictions = teacher_pred_trainer.predict(unlabeled_dataset)

logits = raw_predictions.predictions
probabilities = softmax(logits, axis=1)
pred_labels = np.argmax(probabilities, axis=1)
confidences = np.max(probabilities, axis=1)

df_unlabeled['label'] = pred_labels
df_unlabeled['confidence'] = confidences

THRESHOLD = 0.90
df_pseudo_passed = df_unlabeled[df_unlabeled['confidence'] >= THRESHOLD].copy()

rev_label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
df_manual['label'] = df_manual['label'].map(rev_label_map)
df_pseudo_passed['label'] = df_pseudo_passed['label'].map(rev_label_map)

df_final_10000 = pd.concat([
    df_manual[['cleaned_content', 'label']],
    df_pseudo_passed[['cleaned_content', 'label']]
], ignore_index=True)

df_final_10000.to_csv("dataset_final_10000.csv", index=False, encoding='utf-8')

print(f"Data Manual: {len(df_manual)}")
print(f"Data Pseudo (>=90%): {len(df_pseudo_passed)}")
print(f"Total: {len(df_final_10000)}")

Map:   0%|          | 0/8929 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Data Manual: 976
Data Pseudo (>=90%): 7789
Total: 8765


/tmp/ipykernel_1861/1930275103.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_manual['label'] = df_manual['label'].map(rev_label_map)


### 5. Download Outputs as ZIP

In [25]:
!zip -r teacher_model_and_data.zip ./saved_teacher_model dataset_final_10000.csv
from google.colab import files
files.download('teacher_model_and_data.zip')

  adding: saved_teacher_model/ (stored 0%)
  adding: saved_teacher_model/model.safetensors (deflated 7%)
  adding: saved_teacher_model/tokenizer_config.json (deflated 43%)
  adding: saved_teacher_model/config.json (deflated 58%)
  adding: saved_teacher_model/tokenizer.json (deflated 71%)
  adding: saved_teacher_model/training_args.bin (deflated 54%)
  adding: dataset_final_10000.csv (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>